In [3]:
import pandas as pd
import os

print("Current working directory:")
print(os.getcwd())

Current working directory:
c:\Users\Keshav Sood\OneDrive\Desktop\dermaguard-ai\notebooks


In [4]:
import os

print(os.listdir("../datasets/fitzpatrick17k"))

['background removed', 'fitzpatrick17k.csv']


In [5]:
file_path = '../datasets/fitzpatrick17k/fitzpatrick17k.csv'
df = pd.read_csv(file_path)

print(f"Dataset Shape: {df.shape}")
print(f"Total Rows: {len(df)}")
print(f"Total Columns: {len(df.columns)}")
print("Column Names:")
print(list(df.columns))

df.head()

Dataset Shape: (16577, 9)
Total Rows: 16577
Total Columns: 9
Column Names:
['md5hash', 'fitzpatrick_scale', 'fitzpatrick_centaur', 'label', 'nine_partition_label', 'three_partition_label', 'qc', 'url', 'url_alphanum']


,md5hash,fitzpatrick_scale,fitzpatrick_centaur,label,nine_partition_label,three_partition_label,qc,url,url_alphanum
0,5e82a45bc5d78bd24ae9202d194423f8,3,3,drug induced pigmentary changes,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical-pic/m/minocycline-pigmentation/minocycline-pigmentation1.jpg,httpwwwdermaamincomsiteimagesclinicalpicmminocyclinepigmentationminocyclinepigmentation1jpg.jpg
1,fa2911a9b13b6f8af79cb700937cc14f,1,1,photodermatoses,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical-pic/p/photosensitivity/photosensitivity18.jpg,httpwwwdermaamincomsiteimagesclinicalpicpphotosensitivityphotosensitivity18jpg.jpg
2,d2bac3c9e4499032ca8e9b07c7d3bc40,2,3,dermatofibroma,benign dermal,benign,NaN,https://www.dermaamin.com/site/images/clinical-pic/d/dermatofibroma/dermatofibroma71.jpg,httpwwwdermaamincomsiteimagesclinicalpicddermatofibromadermatofibroma71jpg.jpg
3,0a94359e7eaacd7178e06b2823777789,1,1,psoriasis,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical-pic/p/psoriasis/psoriasis38.jpg,httpwwwdermaamincomsiteimagesclinicalpicppsoriasispsoriasis38jpg.jpg
4,a39ec3b1f22c08a421fa20535e037bba,1,1,psoriasis,inflammatory,non-neoplastic,NaN,https://www.dermaamin.com/site/images/clinical-pic/p/psoriasis-scalp/psoriasis-scalp20.jpg,httpwwwdermaamincomsiteimagesclinicalpicppsoriasisscalppsoriasisscalp20jpg.jpg


In [12]:
# TASK 1 — VERIFY IMAGE FILES
img_dir = '../datasets/fitzpatrick17k/background removed/'
img_files = set(os.listdir(img_dir))
img_extensions = sorted(list(set(os.path.splitext(f)[1].lower() for f in img_files)))

df['image_filename'] = df['md5hash'].astype(str) + '.jpg'
df['image_exists'] = df['image_filename'].isin(img_files)

print(f"Total Image Files in Folder: {len(img_files)}")
print(f"Supported Extensions: {img_extensions}")
print(f"CSV Records WITH Image: {df['image_exists'].sum()}")
print(f"CSV Records WITHOUT Image: {len(df) - df['image_exists'].sum()}")

TASK 1 — VERIFY IMAGE FILES
Total Image Files: 16574
Supported Extensions: ['.jpg']
CSV Records with Image: 16574
CSV Records WITHOUT Image: 3

In [13]:
# TASK 2 — CREATE CLEAN METADATA DATAFRAME
valid_mask = df['image_exists'] & df['three_partition_label'].notnull() & df['label'].notnull()
metadata = df[valid_mask].copy()
metadata['image_path'] = 'datasets/fitzpatrick17k/background removed/' + metadata['image_filename']

selected_cols = [
    'image_path',
    'md5hash',
    'label',
    'three_partition_label',
    'nine_partition_label',
    'fitzpatrick_scale',
    'fitzpatrick_centaur'
]
metadata = metadata[selected_cols].reset_index(drop=True)
print(f"Clean Metadata Shape: {metadata.shape}")
metadata.head()

TASK 2 — CREATE CLEAN METADATA DATAFRAME
Original CSV Rows: 16577
Metadata Clean Rows: 16574
Removed Rows: 3

In [14]:
# TASK 3 — CHECK DUPLICATES
dup_hashes = metadata['md5hash'].duplicated().sum()
dup_paths = metadata['image_path'].duplicated().sum()
dup_rows = metadata.duplicated().sum()

print(f"Duplicate md5hash count: {dup_hashes}")
print(f"Duplicate image_path count: {dup_paths}")
print(f"Duplicate rows count: {dup_rows}")

TASK 3 — CHECK DUPLICATES
Duplicate md5hash: 0
Duplicate image_path: 0
Duplicate rows: 0

In [15]:
# TASK 4 — TARGET LABEL ANALYSIS
print('=== THREE PARTITION LABEL ANALYSIS ===')
three_counts = metadata['three_partition_label'].value_counts()
three_pcts = (metadata['three_partition_label'].value_counts(normalize=True) * 100).round(2)
three_df = pd.DataFrame({'Count': three_counts, 'Percentage (%)': three_pcts})
display(three_df)

print('=== NINE PARTITION LABEL ANALYSIS ===')
nine_counts = metadata['nine_partition_label'].value_counts()
nine_pcts = (metadata['nine_partition_label'].value_counts(normalize=True) * 100).round(2)
nine_df = pd.DataFrame({'Count': nine_counts, 'Percentage (%)': nine_pcts})
display(nine_df)

,Count,Percentage (%)
three_partition_label,,
non-neoplastic,12077,72.87
malignant,2263,13.65
benign,2234,13.48
,Count,Percentage (%)
nine_partition_label,,
inflammatory,10883,65.66
malignant epidermal,1352,8.16
genodermatoses,1194,7.20
benign dermal,1067,6.44


In [16]:
# TASK 5 — SKIN-TONE PRESERVATION
print('=== FITZPATRICK SCALE (PRESERVED AS-IS) ===')
scale_counts = metadata['fitzpatrick_scale'].value_counts(dropna=False)
scale_pcts = (metadata['fitzpatrick_scale'].value_counts(dropna=False, normalize=True) * 100).round(2)
scale_df = pd.DataFrame({'Count': scale_counts, 'Percentage (%)': scale_pcts})
scale_df

,Count,Percentage (%)
fitzpatrick_scale,,
2,4808,29.01
3,3308,19.96
1,2947,17.78
4,2781,16.78
5,1533,9.25
6,635,3.83
-1,562,3.39


In [17]:
# TASK 6 & 7 & 8 — TRAIN/VAL/TEST SPLIT & VERIFICATION
np.random.seed(42)
train_list, val_list, test_list = [], [], []

for label, group in metadata.groupby('three_partition_label'):
    group_shuffled = group.sample(frac=1.0, random_state=42).reset_index(drop=True)
    n = len(group_shuffled)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)
    train_list.append(group_shuffled.iloc[:n_train])
    val_list.append(group_shuffled.iloc[n_train:n_train + n_val])
    test_list.append(group_shuffled.iloc[n_train + n_val:])

train = pd.concat(train_list).sample(frac=1.0, random_state=42).reset_index(drop=True)
validation = pd.concat(val_list).sample(frac=1.0, random_state=42).reset_index(drop=True)
test = pd.concat(test_list).sample(frac=1.0, random_state=42).reset_index(drop=True)

# Save splits to project root datasets folder
metadata.to_csv('../datasets/fitzpatrick17k/metadata_clean.csv', index=False)
train.to_csv('../datasets/fitzpatrick17k/train.csv', index=False)
validation.to_csv('../datasets/fitzpatrick17k/validation.csv', index=False)
test.to_csv('../datasets/fitzpatrick17k/test.csv', index=False)

print(f"Total metadata rows: {len(metadata)}")
print(f"Training rows: {len(train)}")
print(f"Validation rows: {len(validation)}")
print(f"Test rows: {len(test)}")

# Verification of no hash / path overlap
assert len(set(train['md5hash']).intersection(set(validation['md5hash']))) == 0
assert len(set(train['md5hash']).intersection(set(test['md5hash']))) == 0
assert len(set(validation['md5hash']).intersection(set(test['md5hash']))) == 0
print('SUCCESS: Verified 0 hash/path overlap across train, validation, and test splits!')

TASK 6-8 — SPLIT AND VERIFICATION
Total Metadata Rows: 16574
Train Rows: 11600 (69.99%)
Validation Rows: 2485 (14.99%)
Test Rows: 2489 (15.02%)

Hash Overlaps:
Train-Val Overlap: 0
Train-Test Overlap: 0
Val-Test Overlap: 0

,Train (%),Validation (%),Test (%)
three_partition_label,,,
non-neoplastic,72.87,72.88,72.84
malignant,13.66,13.64,13.66
benign,13.47,13.48,13.50


## PREPROCESSING COMPLETE

- **Images Found**: 16574 files in `datasets/fitzpatrick17k/background removed/`
- **Usable Records**: 16574 records (with verified image and target labels)
- **Removed Records**: 3 records (due to missing image files)
- **Training Size**: 11600 samples (70%)
- **Validation Size**: 2485 samples (15%)
- **Test Size**: 2489 samples (15%)
- **Class Distribution (`three_partition_label`)**:
  - `non-neoplastic`: Train 72.87%, Val 72.88%, Test 72.84%
  - `malignant`: Train 13.66%, Val 13.64%, Test 13.66%
  - `benign`: Train 13.47%, Val 13.48%, Test 13.5%
- **Duplicate Status**: 0 duplicate md5hashes, 0 duplicate image paths, 0 duplicate rows.
